In [3]:
%pip install geopandas


  Obtaining dependency information for geopandas from https://files.pythonhosted.org/packages/c4/64/7d344cfcef5efddf9cf32f59af7f855828e9d74b5f862eddf5bfd9f25323/geopandas-1.0.1-py3-none-any.whl.metadata
  Obtaining dependency information for pyogrio>=0.7.2 from https://files.pythonhosted.org/packages/94/8d/24f21e6a93ca418231aee3bddade7a0766c89c523832f29e08a8860f83e6/pyogrio-0.10.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for pyproj>=3.3.0 from https://files.pythonhosted.org/packages/26/0c/b084e8839a117eaad8cb4fbaa81bbb24c6f183de0ee95c6c4e2770ab6f09/pyproj-3.7.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for shapely>=2.0.0 from https://files.pythonhosted.org/packages/b1/5a/6a67d929c467a1973b6bb9f0b00159cc343b02bf9a8d26db1abd2f87aa23/shapely-2.0.6-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/323.6 kB ? eta -:--:--
   ---------------------------------------- 323.6/323.6 kB 6.7 MB/s eta 0:00:00
 

In [52]:
import numpy as np
import pandas as pd
import geopandas as gpd

In [53]:
df_2023 = pd.read_excel('data/SLO County Sales 2023.xlsx', header = 1)

In [54]:
print(df_2023.head())

   Unnamed: 0  Listing ID  S Sub Type      St#          St Name  City  Area  \
0           1  PI22238008  S  CONDO/A  1223 #A       Farroll     ARRG  ARRG   
1           1  PI22244997  S    SFR/D      269     Larchmont     ARRG  ARRG   
2           1  SC22227310  S    SFR/D     2116   El Dorado ST    OSOS  OSOS   
3           1  NS22247014  S  CONDO/A      714  Tanner DR   #C2  PSOR  PRIC   
4           1  PI22223607  S    SFR/D      534       Fein AVE    PSOR  PSOR   

   SLC  L/C Price  Price Per Square Foot      Br/Ba    Sqft  Yr Built  \
0  STD     469000                 469.00  2/1,0,1,0  1000/A  1981/ASR   
1  STD     600000                 428.57  3/2,0,0,0  1400/P  1972/PUB   
2  STD    1030000                 542.11  3/2,0,1,0  1900/O  1986/SLR   
3  STD     360000                 321.43  3/1,0,1,0  1120/A  1977/ASR   
4  STD     575000                 597.71  2/1,0,0,0   962/O  1959/PUB   

       LSqft/Ac Pool Private YN  Garage Spaces Contract Status Change Date  \
0  1,073

In [55]:
area_to_city = {
    "ARRG": "Arroyo Grande",
    "OSOS": "Los Osos",
    "PSOR": "Paso Robles",
    "SLO": "San Luis Obispo",
    "ATSC": "Atascadero",
    "NPMO": "Nipomo",
    "SMIG": "San Miguel",
    "MRBY": "Morro Bay",
    "CAMB": "Cambria",
    "OCNO": "Oceano",
    "SHDN": "Shandon",
    "SMRG": "Santa Margarita",
    "TTON": "Templeton",
    "GRVC": "Grover Beach",
    "PSMO": "Pismo Beach",
    "AVIL": "Avila Beach",
    "BDLY": "Bradley",
    "CAYU": "Cayucos",
    "SMIA": "Santa Maria",
    "SSIM": "San Simeon",
    "CRST": "Creston",
    "CVLL": "Creston Valley"
}

# Use a lambda function with apply to map area codes to city names
df_2023['City'] = df_2023['City'].apply(lambda area: area_to_city.get(area, "Unknown"))

In [58]:
df_2023['St#'] = df_2023['St#'].astype(str).str.strip()
df_2023['St Name'] = df_2023['St Name'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
df_2023['City'] = df_2023['City'].astype(str).str.strip()
df_2023['Full_Address'] = df_2023['Full_Address'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
# Display the Full_Address column to verify
print(df_2023[['St#', 'St Name', 'City', 'Full_Address']].head())


       St#        St Name           City                   Full_Address
0  1223 #A        Farroll  Arroyo Grande  1223 #A Farroll Arroyo Grande
1      269      Larchmont  Arroyo Grande    269 Larchmont Arroyo Grande
2     2116   El Dorado ST       Los Osos     2116 El Dorado ST Los Osos
3      714  Tanner DR #C2    Paso Robles  714 Tanner DR #C2 Paso Robles
4      534       Fein AVE    Paso Robles       534 Fein AVE Paso Robles


In [10]:
%pip install geopy

  Obtaining dependency information for geopy from https://files.pythonhosted.org/packages/e5/15/cf2a69ade4b194aa524ac75112d5caac37414b20a3a03e6865dfe0bd1539/geopy-2.4.1-py3-none-any.whl.metadata
  Obtaining dependency information for geographiclib<3,>=1.52 from https://files.pythonhosted.org/packages/9f/5a/a26132406f1f40cf51ea349a5f11b0a46cec02a2031ff82e391c2537247a/geographiclib-2.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/125.4 kB ? eta -:--:--
   ---------------------------------------- 125.4/125.4 kB 3.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/40.3 kB ? eta -:--:--
   ---------------------------------------- 40.3/40.3 kB 1.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [59]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [60]:
#Get a random sample of 100 homes to test the code: 
df_sample = df_2023.sample(n=10, random_state=1)

In [50]:
def geocode_address(address):
    try:
        location = geolocator.geocode(address)
        if location:
            return pd.Series([location.latitude, location.longitude])
        else:
            return pd.Series([None, None])
    except Exception as e:
        print(f"Error geocoding address {address}: {e}")
        return pd.Series([None, None])

# Apply the geocode function to each address in the 'Full_Address' column with a short delay
df_2023[['Latitude', 'Longitude']] = df_2023['Full_Address'].apply(lambda address: geocode_address(address))

# Display a few rows of the dataframe to confirm
print(df_2023[['Full_Address', 'Latitude', 'Longitude']].head())

Error geocoding address 1223 #A Farroll   , Arroyo Grande, CA: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=1223+%23A+Farroll+++%2C+Arroyo+Grande%2C+CA&format=json&limit=1 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x0000020E4A4E1610>: Failed to establish a new connection: [WinError 10051] A socket operation was attempted to an unreachable network'))
Error geocoding address 269 Larchmont   , Arroyo Grande, CA: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=269+Larchmont+++%2C+Arroyo+Grande%2C+CA&format=json&limit=1 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x0000020E4A4D2210>: Failed to establish a new connection: [WinError 10051] A socket operation was attempted to an unreachable network'))
Error geocoding address 2116 El Dorado ST  , Los Osos, CA: HTTPSConnectionPool(host='nominatim.openstreetm

KeyboardInterrupt: 

In [34]:
# Initialize the geolocator
geolocator = Nominatim(user_agent="geoapi")

In [36]:
# Apply the function to each row and store latitude and longitude
df_sample[['Latitude', 'Longitude']] = df_sample.apply(geocode_address, axis=1)


Error geocoding address 5452 Via Viento   , Atascadero, CA: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=5452+Via+Viento+++%2C+Atascadero%2C+CA&format=json&limit=1 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x0000020E490DE150>: Failed to establish a new connection: [WinError 10051] A socket operation was attempted to an unreachable network'))
Error geocoding address 1950 Newport AVE  , Grover Beach, CA: HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=1950+Newport+AVE++%2C+Grover+Beach%2C+CA&format=json&limit=1 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x0000020E4818CD10>: Failed to establish a new connection: [WinError 10051] A socket operation was attempted to an unreachable network'))
Error geocoding address 254 San Miguel ST  , Avila Beach, CA: HTTPSConnectionPool(host='nominatim.openstreetmap.

692     None
592     None
2381    None
553     None
1603    None
Name: Latitude, dtype: object


In [20]:
area_to_city = {
    "ARRG": "Arroyo Grande",
    "OSOS": "Los Osos",
    "PSOR": "Paso Robles",
    "SLO": "San Luis Obispo",
    "ATSC": "Atascadero",
    "NPMO": "Nipomo",
    "SMIG": "San Miguel",
    "MRBY": "Morro Bay",
    "CAMB": "Cambria",
    "OCNO": "Oceano",
    "SHDN": "Shandon",
    "SMRG": "Santa Margarita",
    "TTON": "Templeton",
    "GRVC": "Grover Beach",
    "PSMO": "Pismo Beach",
    "AVIL": "Avila Beach",
    "BDLY": "Bradley",
    "CAYU": "Cayucos",
    "SMIA": "Santa Maria",
    "SSIM": "San Simeon",
    "CRST": "Creston",
    "CVLL": "Creston Valley"
}

# Add a new 'City' column by mapping area codes from the 'Area' column
df_2023['City'] = df_2023['City'].map(area_to_city)

# Display the updated DataFrame to confirm
print(df_2023[['City']])

     City
0     NaN
1     NaN
2     NaN
3     NaN
4     NaN
...   ...
2377  NaN
2378  NaN
2379  NaN
2380  NaN
2381  NaN

[2382 rows x 1 columns]


# DF 2023

In [61]:

# Initialize Nominatim API with a more descriptive user agent
geolocator = Nominatim(user_agent="my-geocoding-app")

# Apply a rate limiter to avoid exceeding usage limits
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)


In [62]:
#Get a random sample of 100 homes to test the code: 
df_sample = df_2023.sample(n=10, random_state=1)

In [ ]:

address = df_sample['Full_Address']  # Example address

# Get the location with rate limiting
location = geocode(address)

df_sample['Latitude'] = location.latitude
df_sample['Longitude'] = location.longitude

print(f"Coordinates of {address}: ({latitude}, {longitude})")

print(df_sample['City'].unique())

AttributeError: 'NoneType' object has no attribute 'latitude'

In [65]:
def get_coordinates(address):
    location = geocode(address)
    if location:
        return location.latitude, location.longitude
    else:
        return None, None

In [66]:
df_sample[['Latitude', 'Longitude']] = df_sample['Full_Address'].apply(lambda x: pd.Series(get_coordinates(x)))
print(df_sample[['Full_Address','Latitude', 'Longitude']])

In [68]:
df_2023[['Latitude', 'Longitude']] = df_2023['Full_Address'].apply(lambda x: pd.Series(get_coordinates(x)))

RateLimiter caught an error, retrying (0/2 tries). Called with (*('9395 Bridge Canyon WAY San Miguel',), **{}).
Traceback (most recent call last):
  File "c:\Users\matin\anaconda3\Lib\site-packages\urllib3\connectionpool.py", line 466, in _make_request
    six.raise_from(e, None)
  File "<string>", line 3, in raise_from
  File "c:\Users\matin\anaconda3\Lib\site-packages\urllib3\connectionpool.py", line 461, in _make_request
    httplib_response = conn.getresponse()
                       ^^^^^^^^^^^^^^^^^^
  File "c:\Users\matin\anaconda3\Lib\http\client.py", line 1378, in getresponse
    response.begin()
  File "c:\Users\matin\anaconda3\Lib\http\client.py", line 318, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\matin\anaconda3\Lib\http\client.py", line 279, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\matin\anac

In [69]:
print("Missing values in each column:")
print(df_2023[['Latitude', 'Longitude']].isna().sum())

Missing values in each column:
Latitude     655
Longitude    655
dtype: int64


## DF 2024

In [78]:
df_2024 = pd.read_excel('data/SLO County Sales 2024 (1.1 - 9.30).xlsx', header = 1)



In [79]:
df_2024.head()

,Unnamed: 0,Listing ID,S,Sub Type,St#,St Name,City,Area,SLC,L/C Price,Price Per Square Foot,Br/Ba,Sqft,Yr Built,LSqft/Ac,Pool Private YN,Garage Spaces,Contract Status Change Date,DOM/CDOM
0,1,PI23219503,S,SFR/D,411,Ramona Avenue,GRVC,699,STD,605000,568.61,"2/1,0,0,0",1064/P,1946/PUB,"5,000/0.1148",N,2.0,01/02/24,11/45
1,1,NS23182334,S,CONDO/A,5580,Traffic WAY #9,ATSC,ATSC,STD,310000,387.50,"2/1,0,0,0",800/A,1979/ASR,800/0.0184,N,0.0,01/02/24,52/52
2,1,SC23208374,S,SFR/D,486,Bernardo,MRBY,MRBY,"STD,TRUS",950000,730.77,"2/1,1,0,0",1300/O,1987/SLR,"5,581/0.1281",N,2.0,01/02/24,29/29
3,1,SC23209844,S,SFR/D,2618,Rodman DR,OSOS,OSOS,STD,1235000,671.56,"3/2,0,0,0",1839/A,1973/ASR,"9,000/0.2066",N,2.0,01/02/24,5/5
4,1,NS23177685,S,SFR/D,4880,Glenhill LN,PSOR,PRSE,STD,1410000,704.30,"3/2,0,0,0",2002/P,2003/PUB,"426,452/9.79",Y,3.0,01/02/24,50/50


In [81]:
area_to_city = {
    "ARRG": "Arroyo Grande",
    "OSOS": "Los Osos",
    "PSOR": "Paso Robles",
    "SLO": "San Luis Obispo",
    "ATSC": "Atascadero",
    "NPMO": "Nipomo",
    "SMIG": "San Miguel",
    "MRBY": "Morro Bay",
    "CAMB": "Cambria",
    "OCNO": "Oceano",
    "SHDN": "Shandon",
    "SMRG": "Santa Margarita",
    "TTON": "Templeton",
    "GRVC": "Grover Beach",
    "PSMO": "Pismo Beach",
    "AVIL": "Avila Beach",
    "BDLY": "Bradley",
    "CAYU": "Cayucos",
    "SMIA": "Santa Maria",
    "SSIM": "San Simeon",
    "CRST": "Creston",
    "CVLL": "Creston Valley"
}

# Use a lambda function with apply to map area codes to city names
df_2024['City'] = df_2024['City'].map(area_to_city)

In [82]:
df_2024['St#'] = df_2024['St#'].astype(str).str.strip()
df_2024['St Name'] = df_2024['St Name'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
df_2024['City'] = df_2024['City'].astype(str).str.strip()

# Re-create the 'Full_Address' column using the cleaned columns
df_2024['Full_Address'] = df_2024['St#'] + ' ' + df_2024['St Name'] + ', ' + df_2024['City'] 

# Display the Full_Address column to verify
print(df_2024[['St#', 'St Name', 'City', 'Full_Address']].head())

    St#         St Name          City                     Full_Address
0   411   Ramona Avenue  Grover Beach  411 Ramona Avenue, Grover Beach
1  5580  Traffic WAY #9    Atascadero  5580 Traffic WAY #9, Atascadero
2   486        Bernardo     Morro Bay          486 Bernardo, Morro Bay
3  2618       Rodman DR      Los Osos         2618 Rodman DR, Los Osos
4  4880     Glenhill LN   Paso Robles    4880 Glenhill LN, Paso Robles


In [83]:
df_2024[['Latitude', 'Longitude']] = df_2024['Full_Address'].apply(lambda x: pd.Series(get_coordinates(x)))
print("Missing values in each column:")
print(df_2024[['Latitude', 'Longitude']].isna().sum())

RateLimiter caught an error, retrying (0/2 tries). Called with (*('540 116 Pico AVE #116, San Simeon',), **{}).
Traceback (most recent call last):
  File "c:\Users\matin\anaconda3\Lib\site-packages\urllib3\connectionpool.py", line 466, in _make_request
    six.raise_from(e, None)
  File "<string>", line 3, in raise_from
  File "c:\Users\matin\anaconda3\Lib\site-packages\urllib3\connectionpool.py", line 461, in _make_request
    httplib_response = conn.getresponse()
                       ^^^^^^^^^^^^^^^^^^
  File "c:\Users\matin\anaconda3\Lib\http\client.py", line 1378, in getresponse
    response.begin()
  File "c:\Users\matin\anaconda3\Lib\http\client.py", line 318, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\matin\anaconda3\Lib\http\client.py", line 279, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\matin\anac

Missing values in each column:
Latitude     454
Longitude    454
dtype: int64


# Old Code

In [20]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Initialize Nominatim API with a descriptive user agent
geolocator = Nominatim(user_agent="my-geocoding-app")

# Apply a rate limiter to avoid exceeding usage limits
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Apply the geocode directly within the lambda function with error handling
def safe_geocode(address):
    try:
        location = geocode(address)
        if location:
            return location.latitude, location.longitude
        else:
            return None, None
    except GeocoderTimedOut:
        return None, None

# Apply the function to each address in the DataFrame
data[['Latitude', 'Longitude']] = data['Full_Address'].apply(
    lambda x: pd.Series(safe_geocode(x))
)

# Print the updated DataFrame
print(data.head())

RateLimiter caught an error, retrying (0/2 tries). Called with (*('171 Brisco RD   #6, Arroyo Grande',), **{}).
Traceback (most recent call last):
  File "C:\Users\matin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\geopy\adapters.py", line 298, in get_text
    page = self.urlopen(req, timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.1264.0_x64__qbz5n2kfra8p0\Lib\urllib\request.py", line 515, in open
    response = self._open(req, data)
               ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.1264.0_x64__qbz5n2kfra8p0\Lib\urllib\request.py", line 532, in _open
    result = self._call_chain(self.handle_open, protocol, protocol +
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Pyt

NameError: name 'GeocoderTimedOut' is not defined

In [39]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Initialize Nominatim API with a descriptive user agent
geolocator = Nominatim(user_agent="my-geocoding-app")

# Apply a rate limiter to avoid exceeding usage limits
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

In [40]:
address = "2994 Morgan Dr, San Ramon, CA"

location = geocode(address)
latitude = location.latitude
print(f"({latitude}")

#print(data['Full_Address'].head)

location_new = geocode(data['Full_Address'])
latitude_new = location_new.latitude
longitude_new = location_new.longitude
print(f"({latitude_new})")



RateLimiter swallowed an error after 2 retries. Called with (*('2994 Morgan Dr, San Ramon, CA',), **{}).
Traceback (most recent call last):
  File "c:\Users\matin\anaconda3\Lib\site-packages\urllib3\connection.py", line 174, in _new_conn
    conn = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\matin\anaconda3\Lib\site-packages\urllib3\util\connection.py", line 95, in create_connection
    raise err
  File "c:\Users\matin\anaconda3\Lib\site-packages\urllib3\util\connection.py", line 85, in create_connection
    sock.connect(sa)
OSError: [WinError 10051] A socket operation was attempted to an unreachable network

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\matin\anaconda3\Lib\site-packages\urllib3\connectionpool.py", line 714, in urlopen
    httplib_response = self._make_request(
                       ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\matin\anaconda3\Lib\site-package

AttributeError: 'NoneType' object has no attribute 'latitude'

In [43]:
data[['Latitude', 'Longitude']] = data['Full_Address'].apply(lambda address: pd.Series(
    (lambda loc: (loc.latitude, loc.longitude) if loc else (None, None))(
        geocode(address)
    )
))

RateLimiter caught an error, retrying (0/2 tries). Called with (*('1155 Ash ST   #A, Arroyo Grande',), **{}).
Traceback (most recent call last):
  File "C:\Users\matin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\geopy\adapters.py", line 298, in get_text
    page = self.urlopen(req, timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.1264.0_x64__qbz5n2kfra8p0\Lib\urllib\request.py", line 515, in open
    response = self._open(req, data)
               ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.1264.0_x64__qbz5n2kfra8p0\Lib\urllib\request.py", line 532, in _open
    result = self._call_chain(self.handle_open, protocol, protocol +
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Pytho

In [57]:
data.to_csv('adjusted_data_new.csv', index = False)

In [60]:
print(data['Longitude'].isna().sum())

2642


In [17]:
NA_data = pd.read_csv('data/NA_long_lat.csv')

print(NA_data.head())

   Unnamed: 0  Unnamed..0  ...1  Listing.ID  S Sub.Type   St.  \
0           1           5     1  ML81768077  S      SFR  4923   
1           2           6     1  WS19253758  S   MANL/D   336   
2           3           7     1  WS19241106  S   MANL/D   237   
3           4           8     1  ML81769513  S    SFR/D  4923   
4           5          10     1  NS18293487  S    SFR/D  4040   

           St.Name         City Area  ...  Year  observation_date  \
0  Sparrow Hawk Ln  Paso Robles  699  ...  2019        2019-01-01   
1      Bobwhite DR  Paso Robles  699  ...  2019        2019-01-01   
2          Lema DR       Nipomo  699  ...  2019        2019-01-01   
3  Sparrow Hawk LN  Paso Robles  699  ...  2019        2019-01-01   
4    Soda RD   #C7  Paso Robles  699  ...  2019        2019-01-01   

   ATNHPIUS06079A    HPI Adjustment_Factor Adjusted_Price  \
0           227.5  227.5               1.0       206000.0   
1           227.5  227.5               1.0       285000.0   
2          

In [18]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Initialize Nominatim API with a descriptive user agent
geolocator = Nominatim(user_agent="my-geocoding-app")

# Apply a rate limiter to avoid exceeding usage limits
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)




In [19]:
NA_data[['Latitude', 'Longitude']] = NA_data['Full_Address'].apply(lambda address: pd.Series(
    (lambda loc: (loc.latitude, loc.longitude) if loc else (None, None))(
        geocode(address)
    )
))



RateLimiter caught an error, retrying (0/2 tries). Called with (*("1165 Maple ST   #' I ', Arroyo Grande",), **{}).
Traceback (most recent call last):
  File "C:\Users\matin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\geopy\adapters.py", line 298, in get_text
    page = self.urlopen(req, timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.1264.0_x64__qbz5n2kfra8p0\Lib\urllib\request.py", line 515, in open
    response = self._open(req, data)
               ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.1264.0_x64__qbz5n2kfra8p0\Lib\urllib\request.py", line 532, in _open
    result = self._call_chain(self.handle_open, protocol, protocol +
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation

In [16]:
NA_data.to_csv('NA_data.csv')

In [23]:
print(NA_data['Longitude'].isna().sum())


2627


In [85]:
df_2023.to_csv("data/Cleaned_SLO County Sales 2023.csv", index = False)
df_2024.to_csv("data/Cleaned_SLO County Sales 2024.csv", index = False)